# Step 6 — Full Pipeline: Label → Train → Evaluate

Runs the complete pipeline:
1. Load and resample data
2. Smooth + find extrema
3. Label Double Bottom windows
4. Time-based train/val/test split
5. Train TCN
6. Evaluate on test set

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import torch

from src.data.loader    import load_raw_ohlcv
from src.data.resampler import resample_ohlcv
from src.labeling.smoother      import smooth_ohlcv
from src.labeling.extrema       import find_extrema
from src.labeling.double_bottom import label_double_bottom, windows_to_arrays
from src.training.dataset import time_split, make_dataloaders
from src.models.tcn       import build_model
from src.training.trainer import train, evaluate_test

In [ ]:
with open('../configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print('Device:', device)

## 1. Data pipeline

In [ ]:
df_1min  = load_raw_ohlcv(cfg)
df_15min = resample_ohlcv(df_1min, cfg)
df_smooth, h = smooth_ohlcv(df_15min, cfg)
extrema      = find_extrema(df_smooth, cfg)
labeled      = label_double_bottom(df_smooth, extrema, cfg)

print(f'\nTotal labeled windows: {len(labeled)}')
print(f'  Positive: {sum(w.label==1 for w in labeled)}')
print(f'  Negative: {sum(w.label==0 for w in labeled)}')

## 2. Convert to arrays and split

In [ ]:
# Sort by anchor bar (time order) before splitting
labeled_sorted = sorted(labeled, key=lambda w: w.anchor_bar)

X, y = windows_to_arrays(labeled_sorted, df_smooth, cfg)
print('X shape:', X.shape, '  y shape:', y.shape)
print('Label distribution:', dict(zip(*[v.tolist() for v in np.unique(y, return_counts=True)])))

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = time_split(X, y, cfg)

train_loader, val_loader, test_loader = make_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test, cfg
)

## 3. Build model

In [ ]:
model = build_model(cfg, device)
print(model)

## 4. Train

In [ ]:
history = train(model, train_loader, val_loader, cfg, device)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='train loss')
axes[0].plot(history['val_loss'],   label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_f1'], label='val F1', color='green')
axes[1].set_title('Validation F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('TCN Training — Double Bottom')
plt.tight_layout()
plt.show()

## 5. Test set evaluation

In [ ]:
test_metrics = evaluate_test(model, test_loader, cfg, device)
test_metrics